# Factuality Evaluation: FActScore and Claim-Level Verification

LLMs confidently generate false information at a rate that makes aggregate accuracy metrics misleading. Factuality evaluation requires decomposing outputs into atomic claims and verifying each claim independently — a fundamentally different approach from n-gram overlap or preference scoring.

## The Hallucination Problem

Hallucination in LLMs refers to generated content that is fluent and confident but factually incorrect. It exists on a spectrum:
- **Intrinsic hallucination**: contradicts information in the prompt
- **Extrinsic hallucination**: fabricates information not inferable from the prompt
- **Conflation**: blends details from multiple real entities (common in biography generation)

The community discovered that standard metrics completely miss this. A hallucinatory summary can achieve high ROUGE scores because the false claims often share structure with true ones.

## FActScore: Atomic Claim Decomposition

FActScore (Min et al. 2023) proposed a principled framework: decompose a long-form generation into atomic facts, then verify each fact against a knowledge source using an NLI model. The FActScore is the fraction of atomic claims supported by the source.

This approach:
1. Makes evaluation interpretable — you can see exactly which claims are wrong
2. Scales to long-form content that holistic metrics cannot handle
3. Separates factuality from fluency — you can have a perfectly fluent, well-organized, mostly-false response

Limitation: claim decomposition and NLI verification each introduce error. FActScore is only as good as the decomposer and verifier.

In [ ]:
from dataclasses import dataclass
import re

@dataclass
class FactResult:
    claims: list
    supported: list
    not_supported: list
    factscore: float
    source_used: str

def decompose(text: str) -> list:
    # Production: use an LLM to decompose into atomic claims
    # Here: split on sentence boundaries as a rough proxy
    sentences = re.split(r'(?<=[.!?]) +', text.strip())
    claims = []
    for sent in sentences:
        sent = sent.strip()
        if len(sent) > 15:  # filter fragments
            claims.append(sent)
    return claims

def retrieve(claim: str, knowledge_base: dict) -> list:
    # Production: use a retrieval system (BM25, dense retrieval)
    # Here: keyword match against known facts
    claim_words = set(claim.lower().split())
    relevant = []
    for key, facts in knowledge_base.items():
        key_words = set(key.lower().split())
        if len(claim_words & key_words) >= 2:
            relevant.extend(facts)
    return relevant

def mock_nli(claim: str, evidence: list) -> str:
    # Production: use a trained NLI model (MNLI, TRUE, etc.)
    # Here: simple keyword overlap heuristic
    if not evidence:
        return 'not_supported'
    claim_words = set(claim.lower().split())
    for e in evidence:
        e_words = set(e.lower().split())
        overlap = len(claim_words & e_words) / len(claim_words) if claim_words else 0
        if overlap > 0.4:
            return 'supported'
    return 'not_supported'

def evaluate_factuality(generation: str, knowledge_base: dict) -> FactResult:
    claims = decompose(generation)
    supported = []
    not_supported = []
    for claim in claims:
        evidence = retrieve(claim, knowledge_base)
        verdict = mock_nli(claim, evidence)
        if verdict == 'supported':
            supported.append(claim)
        else:
            not_supported.append(claim)
    n = len(claims)
    factscore = len(supported) / n if n > 0 else 0.0
    return FactResult(claims=claims, supported=supported,
                      not_supported=not_supported, factscore=factscore,
                      source_used='knowledge_base')

kb = {
    'Marie Curie': [
        'Marie Curie was born in Warsaw Poland in 1867.',
        'Marie Curie won two Nobel Prizes.',
        'Marie Curie was the first woman to win a Nobel Prize.',
        'Marie Curie died in 1934 from aplastic anemia.',
    ]
}

# Generation with one hallucination
generation = ('Marie Curie was born in Warsaw Poland in 1867. '
              'She won two Nobel Prizes in both physics and chemistry. '
              'Curie was the first woman to win a Nobel Prize. '
              'She was also the first female professor at MIT.')

result = evaluate_factuality(generation, kb)
print(f'FActScore: {result.factscore:.2%}')
print(f'\nSupported claims ({len(result.supported)}):')
for c in result.supported:
    print(f'  [OK] {c}')
print(f'\nNot supported ({len(result.not_supported)}):')
for c in result.not_supported:
    print(f'  [??] {c}')

## Production Factuality Pipelines

At scale, factuality evaluation involves:
1. **Claim extraction**: LLM-based decomposition into atomic propositions
2. **Retrieval**: dense retrieval against a verified knowledge base (Wikipedia, product documentation)
3. **NLI verification**: fine-tuned model to classify each claim as supported/refuted/neutral
4. **Aggregation**: FActScore per response, with breakdowns by claim type

Tools in production use: FActScore library (original paper code), FactCheck-GPT, SAFE (Search-Augmented Factuality Evaluator from Google), and RAGAS for RAG pipeline factuality.

## Hallucination Types and Mitigations

Different hallucination types require different mitigations:

**Entity hallucination** (wrong names, dates, places): mitigated by RAG — retrieve verified entity information and include in context.

**Reasoning hallucination** (correct premises, wrong conclusion): mitigated by chain-of-thought with step verification.

**Conflation** (mixing attributes of two real entities): mitigated by structured prompting that forces attribute-level generation.

**Citation hallucination** (fabricated references): mitigated by constraining the model to only cite sources present in the context window.

Factuality evaluation should be decomposed by hallucination type, since the remediation strategies are different.